# Predicting the Annual Turnover of a Restaurant 
---

## 1. Problem Statement

**Goal**: The goal of this problem is to predict the Annual Turnover of a restaurant based on the variables provided in the data set. 

**Metric to measure**: The measure of accuracy will be RMSE (Root mean square error)

The predicted Annual Turnover for each restaurant in the Test dataset will be compared with the actual Annual Turnover to calculate the RMSE value of the entire prediction. The lower the RMSE value, the better the model will be.

---


## 2. Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

In [2]:
# -------------------------
# Files + columns (fixed)
# -------------------------
TRAIN_PATH = "Train_dataset_.csv"
TEST_PATH  = "Test_dataset_.csv"

TARGET_COL = "Annual Turnover"
ID_COL     = "Registration Number"

# -------------------------
# 1) Load
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape, "| Test:", test_df.shape)


Train: (3493, 34) | Test: (500, 33)


In [3]:

# -------------------------
# 2) Split features / target
#    Drop ID from features (keep only for submission)
# -------------------------
y = train_df[TARGET_COL]
X = train_df.drop(columns=[TARGET_COL, ID_COL])

X_test = test_df.drop(columns=[ID_COL])
X_test = X_test.reindex(columns=X.columns, fill_value=np.nan)  # align cols



In [4]:
# -------------------------
# 3) Identify categorical columns
# -------------------------
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
cat_features_idx = [X.columns.get_loc(c) for c in cat_cols]

# -------------------------
# 4) Simple missing handling
# -------------------------
for c in cat_cols:
    X[c] = X[c].fillna("Unknown").astype(str)
    X_test[c] = X_test[c].fillna("Unknown").astype(str)

for c in num_cols:
    med = X[c].median()
    X[c] = X[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

print("Features:", X.shape[1], "| Categorical:", len(cat_cols))


Features: 32 | Categorical: 7


In [ ]:

# -------------------------
# 5) Train model (no tuning)
# -------------------------
model = CatBoostRegressor(
    iterations=1200,       # keep small for speed
    learning_rate=0.06,    # faster learning
    depth=8,               # solid default
    l2_leaf_reg=5,         # solid regularization
    subsample=0.8,         # helps generalization
    loss_function="RMSE",
    random_seed=42,
    verbose=200
)

model.fit(X, np.log1p(y), cat_features=cat_features_idx)


0:	learn: 0.5415036	total: 91.2ms	remaining: 1m 49s
200:	learn: 0.3961838	total: 15.6s	remaining: 1m 17s


In [ ]:

# -------------------------
# 6) Predict test + save submission
# -------------------------
pred_log = model.predict(X_test)
pred = np.expm1(pred_log)
pred = np.clip(pred, 0, None)

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL].values,
    TARGET_COL: pred
})

submission.to_csv("submission_catboost.csv", index=False)
print("✔ Saved: submission_catboost.csv")
print(submission.head())


---
## 3. Load the Data

In [ ]:
train = pd.read_csv("Train_dataset_.csv")
test  = pd.read_csv("Test_dataset_.csv")

print(train.shape, test.shape)

train.info()
display(train.head())

In [ ]:
train_df = train.copy()       
test_df = test.copy()   

TARGET = "Annual Turnover"
ID_COL = "Registration Number"
DATE_COL = "Opening Day of Restaurant"
CUISINE_COL = "Cuisine"

print("Train shape:", train_df.shape)

print("Test shape :", test_df.shape)
print("Columns in train:", list(train_df.columns))
print("Columns in test:", list(test_df.columns))

In [ ]:
# Align test columns

train_cols = list(train.columns)
test_cols = list(test.columns)

if train_cols != test_cols:
    print("Column mismatch detected between train and test!")

    # Show differences
    train_only = set(train_cols) - set(test_cols)
    test_only  = set(test_cols) - set(train_cols)

    print("Columns only in TRAIN:", train_only)
    print("Columns only in TEST :", test_only)

    # Auto-fix: align test to train
    test = test.reindex(columns=train_cols)
    print("✔ Test columns realigned to match train.")
    
# Test again
train_cols = list(train.columns)
test_cols = list(test.columns)   
assert train_cols == test_cols, "Train and Test columns still do not match!"
print("Train and Test columns match perfectly.")

In [ ]:
# =========================================
# 2) Hard validation (this catches silent mistakes)
# =========================================
assert TARGET in train_df.columns, f"Target '{TARGET}' not found in train columns!"
assert ID_COL in test_df.columns, f"ID col '{ID_COL}' not found in test!"

# Make sure target is numeric
train_df[TARGET] = pd.to_numeric(train_df[TARGET], errors="coerce")
bad_target = train_df[TARGET].isna().mean()
print(f"Target NaN rate after numeric conversion: {bad_target:.4%}")
train_df = train_df.dropna(subset=[TARGET]).reset_index(drop=True)

# Quick target stats (you NEED to see these)
y = train_df[TARGET].astype(float)
print("\nTarget summary:")
print(y.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print("Any <= 0?:", (y <= 0).any())

# If turnover has zeros/negatives, log1p still works for zero but not for negative.
if (y < 0).any():
    raise ValueError("Target contains negative values; log1p not valid until you fix negatives.")


---
## 4. Cleanup + Feature Split

In [ ]:
# =========================================
# 3) Feature engineering
#    - Date -> age/year/month/dow/weekend
#    - Cuisine -> multi-label multi-hot
# =========================================
def add_date_features(df: pd.DataFrame, date_col: str, asof_date: pd.Timestamp) -> pd.DataFrame:
    df = df.copy()
    if date_col in df.columns:
        dt = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)
        df["open_year"] = dt.dt.year
        df["open_month"] = dt.dt.month
        df["open_dayofweek"] = dt.dt.dayofweek
        df["open_is_weekend"] = df["open_dayofweek"].isin([5, 6]).astype(int)
        df["restaurant_age_days"] = (asof_date - dt).dt.days
        df = df.drop(columns=[date_col])
    return df

# Use max TRAIN opening date as "as-of" anchor (prevents peeking)
asof = pd.to_datetime(train_df[DATE_COL], errors="coerce", dayfirst=True).max()

train_fe = add_date_features(train_df, DATE_COL, asof)
test_fe  = add_date_features(test_df,  DATE_COL, asof)

# Keep a clean copy of y
y = train_fe[TARGET].astype(float)
X = train_fe.drop(columns=[TARGET]).copy()
X_test = test_fe.copy()

# Cuisine handling: multi-hot via CountVectorizer
# We'll build this as a separate transformer and combine with other columns.
def normalize_cuisine(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str).str.lower()
    s = s.str.replace(r"\s*,\s*", ",", regex=True)  # trim spaces around commas
    return s

# If Cuisine col missing, create empty
if CUISINE_COL not in X.columns:
    X[CUISINE_COL] = ""
if CUISINE_COL not in X_test.columns:
    X_test[CUISINE_COL] = ""

X[CUISINE_COL] = normalize_cuisine(X[CUISINE_COL])
X_test[CUISINE_COL] = normalize_cuisine(X_test[CUISINE_COL])

cuisine_vectorizer = CountVectorizer(
    tokenizer=lambda x: x.split(",") if x else [],
    preprocessor=None,
    lowercase=False,
    binary=True,          # multi-hot (presence/absence)
    min_df=2              # drop ultra-rare cuisines that add noise
)

In [ ]:
# =========================================
# 4) Preprocessing (NO scaling for trees; clean OHE for cats)
# =========================================
# Separate main features (excluding cuisine) from cuisine text feature
X_main = X.drop(columns=[CUISINE_COL]).copy()
X_test_main = X_test.drop(columns=[CUISINE_COL]).copy()

num_cols = X_main.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_main.columns if c not in num_cols]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_main = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
], remainder="drop")

In [ ]:
# Full preprocessor: main structured + cuisine multi-hot
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer

prep_structured = Pipeline([
    ("select", FunctionTransformer(lambda df: df, validate=False)),
    ("ct", preprocess_main)
])

prep_cuisine = Pipeline([
    ("select", FunctionTransformer(lambda df: df[CUISINE_COL], validate=False)),
    ("cv", cuisine_vectorizer)
])

full_preprocessor = FeatureUnion([
    ("structured", prep_structured),
    ("cuisine", prep_cuisine)
])


In [ ]:
# =========================================
# 5) Correct evaluation setup
#    - Train models on log1p(target)
#    - Predict back in original space
#    - Score with RMSE in original space (matches leaderboard format)
# =========================================
def rmse(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(y_true, y_pred))

def cv_rmse(model, X_df, y_series, n_splits=5):
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    # we compute RMSE manually to avoid "neg" confusion and ensure it is correct.
    rmses = []
    for tr_idx, va_idx in cv.split(X_df):
        X_tr, X_va = X_df.iloc[tr_idx], X_df.iloc[va_idx]
        y_tr, y_va = y_series.iloc[tr_idx], y_series.iloc[va_idx]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)
        rmses.append(rmse(y_va, pred))
    rmses = np.array(rmses)
    return rmses.mean(), rmses.std()

def wrap_log_target(reg):
    return TransformedTargetRegressor(
        regressor=reg,
        func=np.log1p,
        inverse_func=np.expm1
    )

In [ ]:
# Baseline sanity check: predicting mean (this tells you the natural scale)
mean_pred_rmse = rmse(y, np.full_like(y, y.mean()))
print("\nSanity check RMSE (predict mean):", f"{mean_pred_rmse:,.2f}")
# If your CV RMSE is close to this, you are basically not learning signal.


In [ ]:
# =========================================
# 6) Multiple models
# =========================================
models = {
    "Ridge": Ridge(alpha=10.0, random_state=42),
    "HistGBR": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_leaf_nodes=63,
        min_samples_leaf=20,
        random_state=42
    ),
}

if HAS_LGBM:
    models["LightGBM"] = LGBMRegressor(
        n_estimators=4000,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

if HAS_XGB:
    models["XGBoost"] = XGBRegressor(
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    )

# CatBoost: handle categoricals natively (NO one-hot). We'll use it on raw X (with cuisine as string)
if HAS_CAT:
    # Identify categorical columns for CatBoost (excluding cuisine; cuisine kept as string too)
    cat_cols_cb = [c for c in X.columns if X[c].dtype == "object"]
    models["CatBoost"] = CatBoostRegressor(
        iterations=4000,
        depth=8,
        learning_rate=0.05,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

In [ ]:
# =========================================
# 7) Build pipelines + evaluate + select best
# =========================================
results = []
fitted_pipes = {}

for name, m in models.items():
    if name == "CatBoost":
        # minimal cleaning for CatBoost: fill missing, keep strings
        X_cb = X.copy()
        X_cb[cat_cols_cb] = X_cb[cat_cols_cb].fillna("NA").astype(str)

        # IMPORTANT: log target wrapper here
        est = wrap_log_target(m)

        mean_rmse, std_rmse = cv_rmse(est, X_cb, y, n_splits=5)
        results.append((name, mean_rmse, std_rmse))
        fitted_pipes[name] = ("catboost", est, X_cb)

    else:
        pipe = Pipeline([
            ("prep", full_preprocessor),
            ("model", m)
        ])
        est = wrap_log_target(pipe)

        mean_rmse, std_rmse = cv_rmse(est, X, y, n_splits=5)
        results.append((name, mean_rmse, std_rmse))
        fitted_pipes[name] = ("sklearn", est, X)

    print(f"{name:10s} CV RMSE: {mean_rmse:,.2f} (+/- {std_rmse:,.2f})")
    
rank_df = pd.DataFrame(results, columns=["Model", "CV_RMSE_Mean", "CV_RMSE_Std"]).sort_values("CV_RMSE_Mean")
print("\n=== MODEL RANKING ===")
print(rank_df)

best_name = rank_df.iloc[0]["Model"]
print("\nBest model:", best_name)

## 5. Preprocessing - Handling Missing values (with mean imputation)

In [ ]:
# Column Types
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# For CatBoost: categorical columns must be strings, no NaNs
cat_modes = X[categorical_cols].mode().iloc[0]
X[categorical_cols] = X[categorical_cols].fillna(cat_modes).astype(str)
test_df[categorical_cols] = test_df[categorical_cols].fillna(cat_modes).astype(str)

cat_features_idx = [X.columns.get_loc(col) for col in categorical_cols]



In [ ]:
# PREPROCESSOR (FOR SKLEARN MODELS)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, numerical_cols),
    ("cat", cat_pipe, categorical_cols)
], remainder="drop")



## 6. Model Setup (Strong Baselines)

In [ ]:
# BASE MODELS

models = {
    "LinearRegression": LinearRegression(),
    "ElasticNet": ElasticNet(random_state=42, max_iter=2000),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42),
    "CatBoost": CatBoostRegressor(depth=6, learning_rate=0.05, iterations=300,
                                   verbose=False, random_state=42,
                                   allow_writing_files=False,
                                   cat_features=cat_features_idx),
    "XGBoost": XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8, random_state=42)
}


## 7. Evaluation: K-Fold CV RMSE (No Train Split Needed Here)

In [ ]:
# CV RMSE FUNCTION

def cv_rmse(estimator, X, y, n_splits=5):
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(estimator, X, y,
                              scoring="neg_root_mean_squared_error",
                              cv=cv, n_jobs=-1)
    return -scores.mean(), -scores.std()


## 8. Run cv on all models

In [ ]:
# BASELINE EVALUATION

base_pipes = {}
ranking = []

for name, model in models.items():
    if name == "CatBoost":
        pipe = model
    else:
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])
    base_pipes[name] = pipe

    mean_rmse, std_rmse = cv_rmse(pipe, X, y)
    ranking.append({"Model": name, "CV_RMSE_Mean": mean_rmse, "CV_RMSE_Std": std_rmse})
    print(f"{name:20s} RMSE: {mean_rmse:.2f} (+/- {std_rmse:.2f})")

rank_df = pd.DataFrame(ranking).sort_values("CV_RMSE_Mean")
print("\n=== BASELINE LEADERBOARD ===")
print(rank_df)

top3 = rank_df.head(3)["Model"].tolist()


# Pick the lowest CV_RMSE_Mean. Also check Std — lower std = more stable.
# if error persists, remove HistGradientBoosting with models.pop("HistGradientBoosting", None)

## 9. Hyperparameter Spaces (Top 3 Only)

In [ ]:
param_spaces = {

    "ElasticNet": {
        "model__alpha": loguniform(1e-3, 1e0),
        "model__l1_ratio": np.linspace(0.1, 0.9, 5)
    },

    "HistGradientBoosting": {
        "model__learning_rate": loguniform(0.01, 0.15),
        "model__max_depth": randint(3, 10),
        "model__max_leaf_nodes": randint(20, 120),
        "model__min_samples_leaf": randint(5, 80),
        "model__l2_regularization": loguniform(1e-4, 1.0),
    },

    "LightGBM": {
        "model__n_estimators": randint(200, 600),
        "model__learning_rate": loguniform(0.01, 0.15),
        "model__num_leaves": randint(20, 80),
        "model__max_depth": randint(3, 8),
        "model__min_child_samples": randint(5, 60),
        "model__subsample": np.linspace(0.7, 1.0, 4),
        "model__colsample_bytree": np.linspace(0.7, 1.0, 4),
        "model__reg_alpha": loguniform(1e-4, 1.0),
        "model__reg_lambda": loguniform(1e-4, 1.0),
    },

    # Note: no model__ prefix for CatBoost (not in pipeline)
    "CatBoost": {
        "depth": randint(4, 8),
        "learning_rate": loguniform(0.01, 0.15),
        "iterations": randint(250, 600),
        "l2_leaf_reg": loguniform(1e-2, 3.0),
        "bagging_temperature": loguniform(1e-3, 5.0),
    },

    "XGBoost": {
        "model__n_estimators": randint(200, 600),
        "model__learning_rate": loguniform(0.01, 0.15),
        "model__max_depth": randint(3, 8),
        "model__subsample": np.linspace(0.7, 1.0, 4),
        "model__colsample_bytree": np.linspace(0.7, 1.0, 4),
        "model__gamma": loguniform(1e-4, 1.0),
        "model__reg_alpha": loguniform(1e-4, 1.0),
        "model__reg_lambda": loguniform(1e-4, 1.0),
    },
}



### 9.1. RandomizedSearchCV for Top 3 Models

- Uses 5-fold CV
- Optimises RMSE
- Returns the best tuned pipeline for each model

In [ ]:
cv_tune = KFold(n_splits=3, shuffle=True, random_state=42)

n_iter_map = {
    "ElasticNet": 12,
    "HistGradientBoosting": 12,
    "LightGBM": 12,
    "CatBoost": 10,
    "XGBoost": 12,
}

tuned_best = {}
tuning_results = []

for name in top3:
    print(f"\n=== TUNING {name} ===")

    if name == "CatBoost":
        estimator = CatBoostRegressor(
            verbose=False,
            random_state=42,
            allow_writing_files=False,
            cat_features=cat_features_idx
        )
        param_dist = param_spaces["CatBoost"]
    else:
        estimator = base_pipes[name]
        param_dist = param_spaces[name]

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=n_iter_map[name],
        scoring="neg_root_mean_squared_error",
        cv=cv_tune,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X, y)

    best_rmse = -search.best_score_
    print(f"Best RMSE: {best_rmse:.4f}")
    print("Best Params:", search.best_params_)

    tuned_best[name] = search.best_estimator_
    tuning_results.append({"Model": name, "Tuned_CV_RMSE": best_rmse})

tuned_df = pd.DataFrame(tuning_results).sort_values("Tuned_CV_RMSE")
print("\n=== FINAL TUNED LEADERBOARD ===")
print(tuned_df)

final_model_name = tuned_df.iloc[0]["Model"]
final_model = tuned_best[final_model_name]
print(f"\nSelected Final Model: {final_model_name}")

## 10. Ensemble the Tuned Top 3 (Average Predictions)

In [ ]:
# Prepare test data
X[categorical_cols] = X[categorical_cols].astype("object").fillna("NA")
test_df[categorical_cols] = test_df[categorical_cols].astype("object").fillna("NA")

# Refit best estimators on FULL train
tuned_models = list(tuned_best.values())
for est in tuned_models:
    est.fit(X, y)

# Predict each tuned model on test
pred_matrix = np.column_stack([est.predict(test_df) for est in tuned_models])

# Simple average ensemble
ensemble_preds = pred_matrix.mean(axis=1)
ensemble_preds

## 11. Write Submission (Tuned Ensemble)

In [ ]:
submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: ensemble_preds
})

# Sanity checks: EXACT required format
assert submission.shape[1] == 2, "Submission must have exactly 2 columns."
assert len(submission) == len(test_df), "Submission row count must match test set."
assert ID_COL in submission.columns and TARGET in submission.columns, "Wrong column names."

submission.to_csv("submission_tuned_ensemble.csv", index=False)
print(" Saved: submission_tuned_ensemble.csv")
submission.head()


## 12. Weighted Ensemble (Top 3 tuned models)

In [ ]:
# --- Weighted ensembling of the tuned top models ---

# 1) Build a DataFrame of tuned CV RMSEs
tuned_df = pd.DataFrame(tuning_results).sort_values("Tuned_CV_RMSE")
print(tuned_df)

# 2) Compute weights = inverse RMSE, normalized
inv = 1.0 / tuned_df["Tuned_CV_RMSE"].values
weights = inv / inv.sum()

# Map model -> weight
model_weights = dict(zip(tuned_df["Model"].values, weights))
print("\nWeights:")
for m, w in model_weights.items():
    print(f"{m:20s}: {w:.4f}")

# 3) Refit tuned models on full train (explicit + safe)
for name, est in tuned_best.items():
    est.fit(X, y)

# 4) Get predictions per tuned model on test
preds = []
wts = []

for name in tuned_df["Model"].values:
    est = tuned_best[name]
    preds.append(est.predict(test_df))
    wts.append(model_weights[name])

pred_matrix = np.column_stack(preds)
wts = np.array(wts)

# 5) Weighted average prediction
weighted_preds = pred_matrix @ wts  # (n_samples, n_models) dot (n_models,) -> (n_samples,)

# 6) Save submission
submission_weighted = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: weighted_preds
})

# Sanity checks
assert submission_weighted.shape[1] == 2
assert len(submission_weighted) == len(test_df)

submission_weighted[TARGET] = submission_weighted[TARGET].clip(lower=0)


submission_weighted.to_csv("submission_weighted_tuned_ensemble.csv", index=False)
print("\n Saved: submission_weighted_tuned_ensemble.csv")

submission_weighted.head()
